In [1]:
import numpy as np 
import pandas as pd 
import torch
import ast
import random
import time
import os

In [26]:
def process_and_save_data(file_type):
    # Base path and file name
    base_path = "/home/gridsan/tmackey/cdvae/data/mp_20_dm/"
    file_name = f"{file_type}.csv"

    # Load the DataFrame
    print("reading in the csv ... ") 
    df = pd.read_csv(base_path + file_name)

    # Process the DataFrame
    features = ['xrd_peak_locations', 'xrd_peak_intensities', 'atomic_numbers']
    for feature in features: 
        df[feature] = df[feature].apply(ast.literal_eval)

    df['disc_sim_xrd'] = df['disc_sim_xrd'].apply(lambda x: [float(val) for val in x[1:-1].split() if val])
    df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: list(set(x)))
    df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: random.sample(x, len(x)))

    features.extend(["disc_sim_xrd", "atomic_numbers"])
    for feature in features:
        df[feature] = df[feature].apply(lambda x: (x + [0]*256)[:256])

    # Create tensors
    tensors = {
        'xrd_peak_intensities': torch.stack([torch.tensor(x) for x in df['xrd_peak_intensities']]),
        'xrd_peak_locations': torch.stack([torch.tensor(x) for x in df['xrd_peak_locations']]),
        'atomic_numbers': torch.stack([torch.tensor(x) for x in df['atomic_numbers']]),
        'disc_sim_xrd': torch.stack([torch.tensor(x) for x in df['disc_sim_xrd']])
    }

    # Create dictionaries from tensors
    print("creating dictionaries and saving for ..." ) 
    for key, tensor in tensors.items():
        print(key) 
        tensor_dict = create_dict_from_df_and_tensor(df, tensor)
        torch.save(tensor_dict, f"{base_path}{file_type}_{key}_dict.pt")

def create_dict_from_df_and_tensor(df, tensor):
    assert len(df) == len(tensor)
    result_dict = {}
    for idx, row in df.iterrows():
        materials_id = row['material_id']
        tensor_slice = tensor[idx].unsqueeze(0)  # Get a 1x256 slice of the tensor
        result_dict[materials_id] = tensor_slice
    return result_dict

In [27]:
# Example usage
process_and_save_data("test")

reading in the csv ... 


/state/partition1/slurm_tmp/24590176.0.0/ipykernel_557165/1198615107.py:8: DtypeWarning: Columns (6,8,15,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(base_path + file_name)


creating dictionaries and saving for ...
xrd_peak_intensities
xrd_peak_locations
atomic_numbers
disc_sim_xrd


In [28]:
# Example usage
process_and_save_data("val")

reading in the csv ... 


/state/partition1/slurm_tmp/24590176.0.0/ipykernel_557165/1198615107.py:8: DtypeWarning: Columns (6,8,15,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(base_path + file_name)


creating dictionaries and saving for ...
xrd_peak_intensities
xrd_peak_locations
atomic_numbers
disc_sim_xrd


In [29]:
# Example usage
process_and_save_data("train")

reading in the csv ... 


/state/partition1/slurm_tmp/24590176.0.0/ipykernel_557165/1198615107.py:8: DtypeWarning: Columns (7,9,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(base_path + file_name)


creating dictionaries and saving for ...
xrd_peak_intensities
xrd_peak_locations
atomic_numbers
disc_sim_xrd


Now, let's test to see if we can effectively load in the data

In [31]:
name = "/home/gridsan/tmackey/cdvae/data/mp_20_dm/train"

In [33]:
#read in the dictionaries 
xrd_peak_intensities_dict = torch.load(name + "_xrd_peak_intensities_dict.pt")
xrd_peak_locations_dict = torch.load(name + "_xrd_peak_locations_dict.pt")
atomic_numbers_dict = torch.load(name + "_atomic_numbers_dict.pt")
disc_sim_xrd_dict = torch.load(name + "_disc_sim_xrd_dict.pt")

In [35]:
start = time.time()
xrd_peak_intensities_dict = torch.load(name + "_xrd_peak_intensities_dict.pt")
xrd_peak_locations_dict = torch.load(name + "_xrd_peak_locations_dict.pt")
atomic_numbers_dict = torch.load(name + "_atomic_numbers_dict.pt")
disc_sim_xrd_dict = torch.load(name + "_disc_sim_xrd_dict.pt")
end = time.time()
print("time taken: {}".format(end-start))

time taken: 21.513267517089844


In [2]:

def source_restriction(df, source): 
    if source == "mp_20": 
        try: 
            mp_20_df = df[df['material_id'].str.startswith('m')]
            return mp_20_df
        except:
            print("filter failed")
            return df
    else:
        return df

def compound_restriction(df, name = "batch_82_0_20"): 
    if name == "batch_82_0_20": 
        print("filtering with batch_82_0_20")
        batch_82_0_20 = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20_dm/train_errors/batch_82_0_20.pt')
        num_atoms = batch_82_0_20[0].num_atoms.cpu()
        atom_types = batch_82_0_20[0].atom_types.cpu()

        unflattened_atom_types = unflatten_atom_types(num_atoms, atom_types)
        print("unflattened_atom_types: ", unflattened_atom_types)
        filtered_df = df[~df['atomic_numbers'].isin(unflattened_atom_types)]
        
        return filtered_df
    else: 
        print("no filter")
        return df


In [3]:
def num_atoms_restriction(df, max_num_atoms):
    atomic_num_list = df['atomic_numbers'].apply(ast.literal_eval)
    num_atoms = [len(x) for x in atomic_num_list]
    len_mask = [x<=max_num_atoms for x in num_atoms]
    df = df[len_mask]
    return df

def unflatten_atom_types(num_atoms, atom_types):
    """
    Converts a flattened list of atom types into a list of lists based on num_atoms.

    :param num_atoms: A list or tensor of the number of atoms in each molecule.
    :param atom_types: A flattened list or tensor of atom types.
    :return: A list of lists, where each sublist contains atom types for each molecule.
    """
    # Convert tensors to lists if they are not already
    if not isinstance(num_atoms, list):
        num_atoms = num_atoms.tolist()
    if not isinstance(atom_types, list):
        atom_types = atom_types.tolist()

    result = []
    start = 0
    for count in num_atoms:
        end = start + count
        result.append(str(atom_types[start:end]))
        start = end

    return result

In [31]:
def preprocess(input_file, num_workers, niggli, primitive, graph_method,
               train_fraction, 
               prop_list, max_num_atoms = 20, source = "any"):

    #set constraints
    print("constraints are max_num_atoms = {} and source = {}".format(max_num_atoms, source))

    df_file_name = input_file[:-4] + "_" + str(max_num_atoms) + "_" + source + ".csv"
    graph_file_name = input_file[:-4] + "_" + str(max_num_atoms) + "_" + source + ".pt"

    #look for the file in the directory
    if os.path.isfile(df_file_name) and os.path.isfile(graph_file_name):
        print("using existing csv file ", df_file_name)
        df = pd.read_csv(df_file_name)

        print("using existing graph file ", graph_file_name)
        graph_dict = torch.load(graph_file_name)
    
    else:

        print("creating new csv file ", df_file_name)

        df = pd.read_csv(input_file)

        #impose a restriction on the number of atoms in the compounds being read in 
        #note that this is seperate from the parameter written by Xie - that enforces bounds on the model
        #this limits the data that's being used.
        df = num_atoms_restriction(df, max_num_atoms)
        print("using {} rows after imposing a restriction on the number of atoms".format(len(df)))

        #impose a restriction on the source of the data (used to held deal with combined datasets)
        df = source_restriction(df, source)
        print("using {} rows after imposing a restriction on the source".format(len(df)))

        #save the new dataframe to a csv file
        df.to_csv(df_file_name)
        print("saved dataframe to csv file ", df_file_name)

        #read in the graph pt file
        pt_file_path = input_file[:-4] + ".pt"
        #this will contain the graph data for all the compounds
        graph_dict = torch.load(pt_file_path)

        #make a new dictionary with only entries that have keys in the material_id column of the dataframe
        sub_graph_dict = {key: graph_dict[key] for key in df['material_id'].tolist()}

        #save the new dictionary to a pt file
        torch.save(sub_graph_dict, graph_file_name)
        print("saved graph dictionary to pt file ", graph_file_name)

    #impose a restriction eliminating a few compounds that are causing problems
    df = compound_restriction(df, name = "batch_82_0_20")

    #allow for subsectioning of training data - used for data size impact studies 
    n = round(len(df)*train_fraction)
    if train_fraction < 1:
        df = df.sample(n=n)
    print("using {} rows given a train_fraction of {}".format(n, train_fraction))
    
    #properties stuff, not super important for xrd purposes 
    prop_dictionary = {}
    for prop in prop_list:
        prop_dictionary[prop] = df[prop].values.astype(np.float32)

    try: 
        start = time.time()
        xrd_peak_intensities_dict = torch.load(input_file[:-4] + "_xrd_peak_intensities_dict.pt")
        xrd_peak_locations_dict = torch.load(input_file[:-4] + "_xrd_peak_locations_dict.pt")
        atomic_species_dict = torch.load(input_file[:-4] + "_atomic_numbers_dict.pt")
        disc_sim_xrd_dict = torch.load(input_file[:-4] + "_disc_sim_xrd_dict.pt")
        
        print("xrd, atomic species, and disc sim xrd dicts preloaded")
        
        #merge everything into a list of dictionaries
        ordered_results = []
        for i in range(len(df)):
            materials_id = df['material_id'].iloc[i]
            #all list orders except for graph_arrays should have bene preserved 
            ordered_results.append({'xrd_intensities': xrd_peak_intensities_dict[materials_id][0], #[0] made for dimensionality 
                                    'xrd_locations': xrd_peak_locations_dict[materials_id][0], 
                                    'atomic_species': atomic_species_dict[materials_id][0], 
                                    'disc_sim_xrd': disc_sim_xrd_dict[materials_id][0], 
                                    'graph_arrays': graph_dict[materials_id]})
            for prop in prop_list:
                ordered_results[i][prop] = prop_dictionary[prop][i]
        
        end = time.time()
        print("time taken: {}".format(end-start))
        
    except Exception as e: 
        print(e) 
        
        start = time.time()
        features = ['xrd_peak_locations', 'xrd_peak_intensities', 'atomic_numbers']

        for feature in features: 
            df[feature] = df[feature].apply(ast.literal_eval)

        #disc sim xrd and atomic numbers need special additional steps
        df['disc_sim_xrd'] = df['disc_sim_xrd'].apply(lambda x: [float(val) for val in x[1:-1].split() if val])
        df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: list(set(x)))
        df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: random.sample(x, len(x)))

        features.append("disc_sim_xrd")
        features.append("atomic_numbers")

        for feature in features:
            df[feature] = df[feature].apply(lambda x: (x + [0]*256)[:256])

        xrd_intensities = torch.stack([torch.tensor(x) for x in df['xrd_peak_intensities']])
        xrd_locations = torch.stack([torch.tensor(x) for x in df['xrd_peak_locations']])
        atomic_species = torch.stack([torch.tensor(x) for x in df['atomic_numbers']])
        disc_sim_xrd = torch.stack([torch.tensor(x) for x in df['disc_sim_xrd']])

        #merge everything into a list of dictionaries
        ordered_results = []
        for i in range(len(df)):
            materials_id = df['material_id'].iloc[i]
            #all list orders except for graph_arrays should have bene preserved 
            ordered_results.append({'xrd_intensities': xrd_intensities[i], 'xrd_locations': xrd_locations[i], 
                            'atomic_species': atomic_species[i], 'disc_sim_xrd': disc_sim_xrd[i], 'graph_arrays': graph_dict[materials_id]})
            for prop in prop_list:
                ordered_results[i][prop] = prop_dictionary[prop][i]
    
        end = time.time()
        print("time taken: {}".format(end-start))

    return ordered_results

In [32]:
input_file = "/home/gridsan/tmackey/cdvae/data/mp_20_dm/train.csv"
num_workers = 1
niggli=True
primitive=False
graph_method='crystalnn'
train_fraction = 1
prop_list =  ['formation_energy_per_atom']
max_num_atoms = 20
source = "any"

In [33]:
results_with_preloading = preprocess(input_file, num_workers, niggli, primitive, graph_method,
               train_fraction, 
               prop_list, max_num_atoms = 20, source = "any")

constraints are max_num_atoms = 20 and source = any
using existing csv file  /home/gridsan/tmackey/cdvae/data/mp_20_dm/train_20_any.csv


/state/partition1/slurm_tmp/24590515.0.0/ipykernel_566157/1031106017.py:14: DtypeWarning: Columns (8,10,17,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df_file_name)


using existing graph file  /home/gridsan/tmackey/cdvae/data/mp_20_dm/train_20_any.pt
filtering with batch_82_0_20
unflattened_atom_types:  ['[70, 62, 28, 28, 28, 28, 28, 28, 5, 5, 78, 78]', '[41, 41, 17, 17, 17, 17, 17, 17, 17, 17]', '[38, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 77, 44, 44, 44]', '[66, 66, 66, 39, 14, 14, 14, 14, 76, 76]', '[55, 55, 31, 27, 27, 27, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]', '[66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 43, 43, 43, 78]', '[22, 22, 22, 22, 22, 22, 22, 22, 22, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45]', '[65, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 75, 75]', '[41, 41, 41, 41, 14, 14, 28, 28, 28, 28, 28, 28]', '[39, 39, 39, 90, 46, 46, 46, 46]', '[68, 68, 68, 30, 15, 15, 45, 45, 45]', '[59, 59, 66, 66, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 78, 78, 78, 78, 78, 78]', '[58, 58, 58, 58, 26, 75, 76, 76, 76, 76, 76, 76]', '[65, 90, 5, 5, 5, 5, 5, 5, 77, 77, 77, 77, 77, 77, 77, 77, 77]', '[66, 66, 66, 66, 76, 76, 76, 76, 76, 76, 44, 44

In [29]:
results_without_preloading = preprocess(input_file, num_workers, niggli, primitive, graph_method,
               train_fraction, 
               prop_list, max_num_atoms = 20, source = "any")

constraints are max_num_atoms = 20 and source = any
using existing csv file  /home/gridsan/tmackey/cdvae/data/mp_20_dm/train_20_any.csv


/state/partition1/slurm_tmp/24590515.0.0/ipykernel_566157/2789925599.py:14: DtypeWarning: Columns (8,10,17,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df_file_name)


using existing graph file  /home/gridsan/tmackey/cdvae/data/mp_20_dm/train_20_any.pt
filtering with batch_82_0_20
unflattened_atom_types:  ['[70, 62, 28, 28, 28, 28, 28, 28, 5, 5, 78, 78]', '[41, 41, 17, 17, 17, 17, 17, 17, 17, 17]', '[38, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 77, 44, 44, 44]', '[66, 66, 66, 39, 14, 14, 14, 14, 76, 76]', '[55, 55, 31, 27, 27, 27, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]', '[66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 43, 43, 43, 78]', '[22, 22, 22, 22, 22, 22, 22, 22, 22, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45]', '[65, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 75, 75]', '[41, 41, 41, 41, 14, 14, 28, 28, 28, 28, 28, 28]', '[39, 39, 39, 90, 46, 46, 46, 46]', '[68, 68, 68, 30, 15, 15, 45, 45, 45]', '[59, 59, 66, 66, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 78, 78, 78, 78, 78, 78]', '[58, 58, 58, 58, 26, 75, 76, 76, 76, 76, 76, 76]', '[65, 90, 5, 5, 5, 5, 5, 5, 77, 77, 77, 77, 77, 77, 77, 77, 77]', '[66, 66, 66, 66, 76, 76, 76, 76, 76, 76, 44, 44